In [ ]:
# R script for metadata wrangling
# author: Sheng-Kai Hsu
# date created: 2024.06.05
# data last edited: 2024.08.19
rm(list=ls())
PHYLOGWAS_ROOT <- Sys.getenv("PHYLOGWAS_ROOT", unset = "/workdir/sh2246/p_phyloGWAS")

In [5]:
library(tidyverse)

── Attaching core tidyverse packages ───────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ─────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [ ]:
manualDat = read.csv(file.path(PHYLOGWAS_ROOT, "data/Poaceae_metadata_2024.08.21.csv"),header = T)
limsDat = read.delim(file.path(PHYLOGWAS_ROOT, "data/fullQCData_20240819.txt"),header = T)

In [7]:
manualDat_panand = manualDat[manualDat$source=="PanAnd",]

In [11]:
mergedDat = merge(manualDat_panand,limsDat,by = "tracker_sample_name",all = T)

In [16]:
mergedDat_noNA = mergedDat[!is.na(mergedDat[,1]),]

In [ ]:
dupSample = unique(mergedDat_noNA[duplicated(mergedDat_noNA[,1]),1])

In [21]:
mergedDat_noNAnoDup = mergedDat_noNA[!(mergedDat_noNA[,1]%in%dupSample&mergedDat_noNA$run_id==2|mergedDat_noNA[,2]=="AN20T003"),]

In [ ]:
write.table(mergedDat_noNAnoDup[mergedDat_noNAnoDup$correctSpecies=="Omit",c(1:6,12:14)],
            file.path(PHYLOGWAS_ROOT, "output/metadata_processing/badSampleMetadata.txt"),quote = F,sep = "\t",row.names = F)

In [25]:
mergedDat_noNAnoDupnoBad = mergedDat_noNAnoDup[!mergedDat_noNAnoDup$correctSpecies=="Omit",]

In [ ]:
write.table(mergedDat_noNAnoDupnoBad[with(mergedDat_noNAnoDupnoBad,which(correctSpecies != tracker_organism)),c(1:6,12:14)],
            file.path(PHYLOGWAS_ROOT, "output/metadata_processing/mismatchMetadata.txt"),quote = F,sep = "\t")

In [29]:
limsDat_noNA = limsDat[!is.na(limsDat[,1]),]
dupSample2 = unique(limsDat_noNA[duplicated(limsDat[,1]),1])
limsDat_noNAnoDup = limsDat_noNA[!(limsDat_noNA[,1]%in%dupSample2&limsDat_noNA$run_id==2),]

In [66]:
limsDat_noNAnoDupwithAssemblyID = merge(manualDat_panand[,1:2],limsDat_noNAnoDup,by = 'tracker_sample_name',all.y =  T)

In [70]:
limsDat_noNAnoDupwithAssemblyID = limsDat_noNAnoDupwithAssemblyID[!duplicated(limsDat_noNAnoDupwithAssemblyID$tracker_sample_name),]

In [52]:
manualDat_nonPanAndShort = manualDat[!(manualDat$source%in%"PanAnd"&manualDat$technology%in%"Illumina"),]

In [ ]:
manualDat_nonPanAndShort_noBad = manualDat_nonPanAndShort[!manualDat_nonPanAndShort$correctSpecies%in%c("Omit",""),]

In [73]:
colnames(manualDat_nonPanAndShort_noBad)[c(3,7)] = c("tracker_organism","alternative_names")

In [74]:
outMetadata = rbind(limsDat_noNAnoDupwithAssemblyID[,1:3],manualDat_nonPanAndShort_noBad[,c(1:3)])

In [ ]:
outMetadata[outMetadata==""] = NA
outMetadata = outMetadata[!is.na(outMetadata$assemblyID),]
colnames(outMetadata)[3] = "latest_name"

In [ ]:
write.table(outMetadata,file.path(PHYLOGWAS_ROOT, "data/assembly2spNameMetadata_20240819.txt"),row.names = F,col.names = T,quote = F,sep = "\t")

In [78]:
testMetadata = rbind(limsDat_noNAnoDupwithAssemblyID[,3:4],manualDat_nonPanAndShort_noBad[,c(3,7)])

In [79]:
testMetadata = testMetadata[!duplicated(testMetadata[,1]),]

In [83]:
testMetadata[testMetadata==""] = NA

In [84]:
testMetadata = testMetadata[!duplicated(testMetadata[,1]),]

In [85]:
testMetadata_long <- testMetadata %>%
  separate_rows(alternative_names, sep = ",") %>%
  pivot_longer(cols = everything(), names_to = "original_col", values_to = "names") %>%
  filter(!is.na(names)) %>%
  mutate(latest_name = if_else(original_col == "tracker_organism", names, lag(names, default = first(names)))) %>%
  select(-original_col)


In [ ]:
write.table(testMetadata_long,file.path(PHYLOGWAS_ROOT, "data/spNameMetadata_20240819.txt"),row.names = F,col.names = T,quote = F,sep = "\t")